# Virtualize a few example datasets

1. Grab URLS from CEDA test catalog
2. Virtualize them to OSN bucket
3. Verify that snippet to download them works

In [1]:
import httpx
import icechunk as ic
import xarray as xr
from cmip7_virtualization.virtualize import virtualize_from_urls
from cmip7_virtualization.storage import osn_storage, http_vccs_from_registry
from cmip7_virtualization.catalog import urls_from_stac_item
from cmip7_virtualization.store import repo_exists

In [2]:
BUCKET = 'leap-pangeo-pipeline'
ROOT_PREFIX = 'cmip7-virtualization/icechunk-v2'
CEDA_STAC  = "https://api.stac.esgf.ceda.ac.uk"
COLLECTION = "CMIP6"
N_ITEMS = 4

In [3]:
import subprocess

def op_read(ref):
    return subprocess.check_output(["op", "read", ref]).decode().strip()

key    = op_read("op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Access_Key")
secret = op_read("op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Secret_Access_Key")

In [4]:
r = httpx.get(f"{CEDA_STAC}/collections/{COLLECTION}/items?limit={N_ITEMS}", timeout=30)
r.raise_for_status()
all_items = r.json()["features"]

In [5]:
all_items

[{'type': 'Feature',
  'stac_version': '1.1.0',
  'stac_extensions': ['https://stac-extensions.github.io/alternate-assets/v1.2.0/schema.json',
   'https://stac-extensions.github.io/file/v2.1.0/schema.json',
   'https://esgf.github.io/stac-transaction-api/cmip6/v2.0.0/schema.json'],
  'id': 'CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp585.r1i1p1f2.AERday.zg500.gn.v20190726',
  'collection': 'CMIP6',
  'geometry': {'type': 'Polygon',
   'coordinates': [[[-179.0625, -89.375],
     [179.0625, -89.375],
     [179.0625, 89.375],
     [-179.0625, 89.375],
     [-179.0625, -89.375]]]},
  'bbox': [-179.0625, -89.375, 179.0625, 89.375],
  'properties': {'title': 'CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp585.r1i1p1f2.AERday.zg500.gn',
   'datetime': None,
   'created': '2026-07-23T14:03:11.261332Z',
   'updated': '2026-07-23T14:03:11.267208Z',
   'start_datetime': '2015-01-01T12:00:00Z',
   'end_datetime': '2100-12-30T12:00:00Z',
   'license': 'CC0-1.0',
   'version': '20190726',
   'retracted': False,
  

In [6]:
# import xarray as xr
# ds = xr.open_dataset(
#     "https://dap.ceda.ac.uk/badc/cmip6/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp585/r1i1p1f2/AERday/zg500/gn/v20190726/zg500_AERday_UKESM1-0-LL_ssp585_r1i1p1f2_gn_20150101-20491230.nc",
# engine='h5netcdf')

In [8]:
# from virtualizarr.xarray import open_virtual_mfdataset

# urls = urls_from_stac_item(item)

# open_virtual_dataset(

In [ ]:
written_item_dict = {}
for item in all_items:
    
    cmip_id = item.get('id', False)
    assert cmip_id

    print(f"Creating icechunk storage")
    prefix = f"{ROOT_PREFIX}/{cmip_id}/"
    storage = osn_storage(
        bucket=BUCKET,
        prefix=prefix,
        access_key_id=key,
        secret_access_key=secret
        )
    written_item_dict[cmip_id] = storage
    
    # skip if repo already exists: (for now skip, since the ref building is slow, we can later think about updates?)
    if repo_exists(storage):
        print(f"skipping {prefix=} — already exists")
        continue    
    
    # Build references
    print(f"Building references")
    urls = urls_from_stac_item(item)
    print(f"{urls=}")
    vds, registry = virtualize_from_urls(urls)

    print(f"Setup repo")
    vccs = http_vccs_from_registry(registry=registry)
    config = ic.RepositoryConfig.default()
    for vcc in vccs:
        config.set_virtual_chunk_container(vcc)
    repo = ic.Repository.open_or_create(
        storage=storage,
        config=config
    )
    
    print(f"Write to icechunk \n {prefix=}")
    session = repo.writable_session('main')
    vds.vz.to_icechunk(session.store)
    snapshot_id = session.commit("cmip")
    repo.save_config()
    print(snapshot_id)

Creating icechunk storage
Building references
urls=['https://dap.ceda.ac.uk/badc/cmip6/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp585/r1i1p1f2/AERday/zg500/gn/v20190726/zg500_AERday_UKESM1-0-LL_ssp585_r1i1p1f2_gn_20150101-20491230.nc', 'https://dap.ceda.ac.uk/badc/cmip6/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp585/r1i1p1f2/AERday/zg500/gn/v20190726/zg500_AERday_UKESM1-0-LL_ssp585_r1i1p1f2_gn_20500101-21001230.nc']


/Users/juliusbusecke/Code/cmip7-virtualization/.worktrees/icechunk-2-live-read/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/Users/juliusbusecke/Code/cmip7-virtualization/.worktrees/icechunk-2-live-read/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/Users/juliusbusecke/Code/cmip7-virtualization/.worktrees/icechunk-2-live-read/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/Users/juliusbusecke/Code/cmip7-virtu

Setup repo
Write to icechunk 
 prefix='cmip7-virtualization/icechunk-v2/CMIP6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp585.r1i1p1f2.AERday.zg500.gn.v20190726/'
VTXGE83MGPFAQ63GJGFG
Creating icechunk storage
Building references
urls=['https://dap.ceda.ac.uk/badc/cmip6/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp585/r1i1p1f2/AERday/zg1000/gn/v20210611/zg1000_AERday_UKESM1-0-LL_ssp585_r1i1p1f2_gn_20150101-20491230.nc', 'https://dap.ceda.ac.uk/badc/cmip6/data/CMIP6/ScenarioMIP/MOHC/UKESM1-0-LL/ssp585/r1i1p1f2/AERday/zg1000/gn/v20210611/zg1000_AERday_UKESM1-0-LL_ssp585_r1i1p1f2_gn_20500101-21001230.nc']


/Users/juliusbusecke/Code/cmip7-virtualization/.worktrees/icechunk-2-live-read/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


# snippet to download the data locally (new version -- icechunk v2)
You can download the icechunk stores locally using the AWS CLI

```
aws s3 cp s3://leap-pangeo-pipeline/cmip7-virtualization/icechunk-v2/<prefix>/ \
  ./local-icechunk/ \
  --recursive \
  --no-sign-request \
  --endpoint-url https://nyu1.osn.mghpcc.org
```
with any of the above prefixes, for example:

```
aws s3 cp s3://leap-pangeo-pipeline/cmip7-virtualization/icechunk-v2/CMIP6.VolMIP.NERC.UKESM1-0-LL.volc-pinatubo-full.r9i1p1f2.day.zg.gn.v20230810/ \
  ./local-icechunk/ \
  --recursive \
  --no-sign-request \
  --endpoint-url https://nyu1.osn.mghpcc.org
```

<details>
    <summary>Old location</summary>

# snippet to download the data locally (old version -- icechunk v1)
You can download the icechunk stores locally using the AWS CLI

```
aws s3 cp s3://leap-pangeo-pipeline/cmip7-virtualization/<prefix>/ \
  ./local-icechunk/ \
  --recursive \
  --no-sign-request \
  --endpoint-url https://nyu1.osn.mghpcc.org
```
with any of the above prefixes, for example:

```
aws s3 cp s3://leap-pangeo-pipeline/cmip7-virtualization/CMIP6.VolMIP.NERC.UKESM1-0-LL.volc-pinatubo-full.r9i1p1f2.day.zg.gn.v20230810/ \
  ./local-icechunk/ \
  --recursive \
  --no-sign-request \
  --endpoint-url https://nyu1.osn.mghpcc.org
```

</details>

In [ ]:
storage

In [ ]:
# Test that we can read the data!
repo_read = ic.Repository.open(
    storage=storage,
    authorize_virtual_chunk_access={
        'https://dap.ceda.ac.uk/': None, # hardcoded for now, should be resolved dynamically
    }
)
session = repo_read.readonly_session('main')
ds = xr.open_zarr(session.store)

In [ ]:
ds.ta.mean(['time', 'plev']).plot()